# Hit Point JSON + Overlay Lab

This notebook detects hit/breakpoint candidates from ball trajectory using before/after mean reversal scores, exports JSON, and writes the detected points back onto the source video.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "experiments"} and PROJECT_ROOT.parent.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cv.config import INPUT_DIR, OUTPUT_DIR as CV_OUTPUT_DIR

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 60)


## 1. Input / Output Paths

If a compatible RANSAC CSV exists, this notebook can use `ransac_event` as the detection track. Otherwise it falls back to the main ball track from `ball_coords_raw.json`.


In [2]:
BALL_JSON_PATH = CV_OUTPUT_DIR / "ball_coords_raw.json"
MATCH_FRAMES_JSON_PATH = CV_OUTPUT_DIR / "ball_coords_match_frames.json"
INITIAL_RANSAC_CSV_PATH = CV_OUTPUT_DIR / "winransac_initial_trajectory_candidate.csv"
SEGMENTED_RANSAC_CSV_PATH = CV_OUTPUT_DIR / "winransac_segmented_trajectory_candidate.csv"
TRAJECTORY_CSV_CANDIDATES = [
    INITIAL_RANSAC_CSV_PATH,
    CV_OUTPUT_DIR / "winransac_trajectory_candidate.csv",
    CV_OUTPUT_DIR / "trajectory_optimization_candidate.csv",
]

PREFER_RANSAC_EVENT_TRACK = True
RANSAC_TRACK_COLUMNS = ("ransac_event_x", "ransac_event_y")

HIT_POINTS_JSON_PATH = CV_OUTPUT_DIR / "hit_points_mean_reversal.json"
HIT_OVERLAY_VIDEO_PATH = CV_OUTPUT_DIR / "hit_points_mean_reversal_overlay.mp4"

BALL_JSON_PATH, INITIAL_RANSAC_CSV_PATH, SEGMENTED_RANSAC_CSV_PATH, HIT_POINTS_JSON_PATH, HIT_OVERLAY_VIDEO_PATH

(PosixPath('/Users/mac/PycharmProjects/tennis_data_master/cv/data/output/ball_coords_raw.json'),
 PosixPath('/Users/mac/PycharmProjects/tennis_data_master/cv/data/output/trajectory_optimization_candidate.csv'),
 PosixPath('/Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal.json'),
 PosixPath('/Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal_overlay.mp4'))

## 2. Load Ball Coordinates


In [3]:
def load_ball_payload(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    metadata = payload.get("metadata", {})

    if "ball_coords" in payload:
        rows = payload["ball_coords"]
        frames = []
        for row in rows:
            frames.append({
                "frame_index": int(row.get("frame_index", len(frames))),
                "time_sec": row.get("time_sec"),
                "pixel": row.get("pixel"),
                "confidence": row.get("confidence", row.get("heat_conf")),
            })
    elif "frames" in payload:
        frames = []
        for index, frame in enumerate(payload["frames"]):
            ball = frame.get("ball") or {}
            frames.append({
                "frame_index": int(frame.get("frame_index", index)),
                "time_sec": frame.get("time_sec"),
                "pixel": ball.get("pixel") if isinstance(ball, dict) else None,
                "confidence": ball.get("confidence", ball.get("heat_conf")) if isinstance(ball, dict) else None,
            })
    else:
        raise ValueError("Unsupported input JSON. Expected 'ball_coords' or 'frames'.")

    return payload, metadata, frames


def coords_to_array(coords):
    arr = np.full((len(coords), 2), np.nan, dtype=np.float64)
    for index, point in enumerate(coords):
        if point is None:
            continue
        x, y = point
        if x is None or y is None:
            continue
        arr[index] = [float(x), float(y)]
    return arr


def array_to_coords(arr):
    coords = []
    for x, y in np.asarray(arr, dtype=np.float64):
        if not np.isfinite(x) or not np.isfinite(y):
            coords.append(None)
        else:
            coords.append((float(x), float(y)))
    return coords


payload, metadata, frames = load_ball_payload(BALL_JSON_PATH)
fps = float(metadata.get("fps") or 25.0)
frame_width = int(metadata.get("video_width") or 1920)
frame_height = int(metadata.get("video_height") or 1080)
frame_indices = [int(row["frame_index"]) for row in frames]
raw_coords = [tuple(row["pixel"]) if row.get("pixel") else None for row in frames]
raw_confidences = [float(row["confidence"]) if row.get("confidence") is not None else np.nan for row in frames]
raw_arr = coords_to_array(raw_coords)

print(f"loaded frames={len(frames):,}, detections={np.isfinite(raw_arr[:, 0]).sum():,}, fps={fps}, size={frame_width}x{frame_height}")
metadata


loaded frames=929, detections=731, fps=60.0, size=1920x1080


{'version': 'ball-coords-1.0.0',
 'source_video': 'long4.mp4',
 'source_video_path': '/Users/mac/PycharmProjects/tennis_data_master/cv/data/input/long4.mp4',
 'ball_model_path': '/Users/mac/PycharmProjects/tennis_data_master/cv/models/weights/model_best.pt',
 'fps': 60.0,
 'video_width': 1920,
 'video_height': 1080,
 'total_frames': 929,
 'generated_at': '2026-07-05T07:20:05.333560+00:00'}

## 3. Choose Detection Track

Use `ransac_event` when available, because hit/breakpoint detection should run on a cleaner event-preserving track. If it is not available, use the main ball track from `ball_coords_raw.json`.


In [4]:
def _track_csv_matches(track_df):
    x_col, y_col = RANSAC_TRACK_COLUMNS
    if x_col not in track_df.columns or y_col not in track_df.columns:
        return False
    if len(track_df) != len(raw_coords):
        return False
    if "frame" in track_df.columns:
        frame_values = track_df["frame"].to_numpy()
        if len(frame_values) == len(frame_indices) and not np.array_equal(frame_values, np.asarray(frame_indices)):
            return False
    return True


def load_detection_track():
    if PREFER_RANSAC_EVENT_TRACK:
        for csv_path in TRAJECTORY_CSV_CANDIDATES:
            if not csv_path.exists():
                continue
            track_df = pd.read_csv(csv_path)
            if _track_csv_matches(track_df):
                x_col, y_col = RANSAC_TRACK_COLUMNS
                arr = track_df[[x_col, y_col]].to_numpy(dtype=np.float64)
                return arr, f"trajectory_csv:{csv_path.name}:ransac_event"
            print(f"{csv_path.name} exists but ransac_event columns/length are not compatible; trying next source")
    return raw_arr.copy(), "ball_json:main_ball"


track_arr, track_source = load_detection_track()
track_coords = array_to_coords(track_arr)
print("track_source:", track_source)
pd.DataFrame({
    "track_source": [track_source],
    "valid_points": [int(np.isfinite(track_arr[:, 0]).sum())],
    "frames": [len(track_arr)],
})

track_source: trajectory_csv:ransac_event


,track_source,valid_points,frames
0,trajectory_csv:ransac_event,814,929


## 4. Mean-Reversal Hit Candidate Detection

This mirrors the existing hit logic: compare the current point with the previous/next rolling means on x and y. Here we do not require player boxes, so the result should be interpreted as hit/breakpoint candidates.


In [5]:
def mean_motion_vector(points):
    points = np.asarray(points, dtype=np.float64)
    if len(points) < 2 or not np.all(np.isfinite(points)):
        return np.array([np.nan, np.nan], dtype=np.float64)
    steps = np.diff(points, axis=0)
    return np.nanmean(steps, axis=0)


def calculate_turn_angle(track_arr, center_index, *, prev_window=3, next_window=3):
    prev_points = track_arr[center_index - prev_window:center_index]
    next_points = track_arr[center_index + 1:center_index + 1 + next_window]
    v_in = mean_motion_vector(prev_points)
    v_out = mean_motion_vector(next_points)
    if not np.all(np.isfinite(v_in)) or not np.all(np.isfinite(v_out)):
        return np.nan
    norm_in = np.linalg.norm(v_in)
    norm_out = np.linalg.norm(v_out)
    if norm_in < 1e-6 or norm_out < 1e-6:
        return np.nan
    cosine = np.clip(np.dot(v_in, v_out) / (norm_in * norm_out), -1.0, 1.0)
    return math.degrees(math.acos(cosine))


def perspective_score_scale(point_y, frame_h, *, far_scale=0.65, near_scale=2.30):
    if frame_h <= 0 or point_y is None or not np.isfinite(point_y):
        return np.nan
    y_ratio = float(np.clip(point_y / frame_h, 0.0, 1.0))
    return far_scale + y_ratio * (near_scale - far_scale)


def build_mean_reversal_table(
    track_arr,
    confidences,
    *,
    prev_window=3,
    next_window=3,
    far_score_scale=0.65,
    near_score_scale=2.30,
):
    df = pd.DataFrame({
        "x": track_arr[:, 0],
        "y": track_arr[:, 1],
        "confidence": np.asarray(confidences, dtype=np.float64),
    })
    df["prev_y_mean"] = df["y"].rolling(window=prev_window, min_periods=prev_window).mean().shift(1)
    df["next_y_mean"] = df["y"].iloc[::-1].rolling(window=next_window, min_periods=next_window).mean().shift(1).iloc[::-1]
    df["prev_x_mean"] = df["x"].rolling(window=prev_window, min_periods=prev_window).mean().shift(1)
    df["next_x_mean"] = df["x"].iloc[::-1].rolling(window=next_window, min_periods=next_window).mean().shift(1).iloc[::-1]
    df["score_y"] = (df["y"] - df["prev_y_mean"]) + (df["y"] - df["next_y_mean"])
    df["score_x"] = (df["x"] - df["prev_x_mean"]) + (df["x"] - df["next_x_mean"])
    df["abs_total_score"] = df["score_y"].abs() + df["score_x"].abs()
    df["perspective_score_scale"] = df["y"].apply(
        lambda y: perspective_score_scale(y, frame_height, far_scale=far_score_scale, near_scale=near_score_scale)
    )
    df["adjusted_abs_score"] = df["abs_total_score"] / df["perspective_score_scale"]
    angles = np.full(len(df), np.nan, dtype=np.float64)
    for center_index in range(prev_window, len(df) - next_window):
        angles[center_index] = calculate_turn_angle(
            track_arr,
            center_index,
            prev_window=prev_window,
            next_window=next_window,
        )
    df["angle_deg"] = angles
    return df


def group_candidates(candidates, *, group_gap=12):
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda row: row["frame_index"])
    groups = [[candidates[0]]]
    for item in candidates[1:]:
        if item["frame_index"] - groups[-1][-1]["frame_index"] < group_gap:
            groups[-1].append(item)
        else:
            groups.append([item])
    final_hits = []
    for group in groups:
        # Confidence is useful but should not dominate; low-confidence spikes still need inspection.
        best = max(group, key=lambda row: row["adjusted_abs_score"] * (0.7 + 0.3 * row.get("confidence_for_score", 0.5)))
        best = dict(best)
        best["group_size"] = len(group)
        final_hits.append(best)
    return final_hits


def detect_mean_reversal_hits(
    track_arr,
    confidences,
    frame_indices,
    *,
    prev_window=3,
    next_window=3,
    min_abs_total_score=8.0,
    min_angle_deg=30,
    far_score_scale=0.65,
    near_score_scale=2.30,
    min_confidence=0.0,
    group_gap=9,
):
    df = build_mean_reversal_table(
        track_arr,
        confidences,
        prev_window=prev_window,
        next_window=next_window,
        far_score_scale=far_score_scale,
        near_score_scale=near_score_scale,
    )
    candidates = []
    for row_index in range(max(prev_window, next_window), len(df) - max(prev_window, next_window)):
        row = df.iloc[row_index]
        if not np.isfinite(row["x"]) or not np.isfinite(row["y"]):
            continue
        if not np.isfinite(row["score_y"]) or not np.isfinite(row["score_x"]):
            continue
        if not np.isfinite(row["adjusted_abs_score"]):
            continue
        if not np.isfinite(row["angle_deg"]):
            continue
        confidence = float(row["confidence"]) if np.isfinite(row["confidence"]) else np.nan
        if np.isfinite(confidence) and confidence < min_confidence:
            continue

        if row["adjusted_abs_score"] < min_abs_total_score:
            continue
        if row["angle_deg"] < min_angle_deg:
            continue

        candidates.append({
            "row_index": int(row_index),
            "frame_index": int(frame_indices[row_index]),
            "time_sec": round(float(frame_indices[row_index] / fps), 4),
            "pixel": [round(float(row["x"]), 2), round(float(row["y"]), 2)],
            "confidence": round(confidence, 4) if np.isfinite(confidence) else None,
            "confidence_for_score": float(np.clip(confidence, 0.0, 1.0)) if np.isfinite(confidence) else 0.5,
            "score_x": round(float(row["score_x"]), 4),
            "score_y": round(float(row["score_y"]), 4),
            "abs_total_score": round(float(row["abs_total_score"]), 4),
            "perspective_score_scale": round(float(row["perspective_score_scale"]), 4),
            "adjusted_abs_score": round(float(row["adjusted_abs_score"]), 4),
            "angle_deg": round(float(row["angle_deg"]), 4),
        })

    final_hits = group_candidates(candidates, group_gap=group_gap)
    final_frames = {hit["frame_index"] for hit in final_hits}
    for item in candidates:
        item["is_final_hit"] = item["frame_index"] in final_frames
    for item in final_hits:
        item["is_final_hit"] = True
    return df, candidates, final_hits


HIT_DETECT_CONFIG = {
    "prev_window": 3,
    "next_window": 3,
    "min_abs_total_score": 8.0,
    "min_angle_deg": 10.0,
    "far_score_scale": 0.65,
    "near_score_scale": 2.30,
    "min_confidence": 0.0,
    "group_gap": 12,
}

score_df, candidate_hits, final_hits = detect_mean_reversal_hits(
    track_arr,
    raw_confidences,
    frame_indices,
    **HIT_DETECT_CONFIG,
)

print(f"candidate_hits={len(candidate_hits)}, final_hits={len(final_hits)}")
pd.DataFrame(final_hits).head(30)


candidate_hits=149, final_hits=28


,row_index,frame_index,time_sec,pixel,confidence,confidence_for_score,score_x,score_y,abs_total_score,perspective_score_scale,adjusted_abs_score,angle_deg,group_size,is_final_hit
0,8,8,0.1333,"[1032.79, 149.69]",NaN,0.5000,-3.0258,-20.9664,23.9922,0.8787,27.3043,26.3979,1,True
1,41,41,0.6833,"[798.0, 744.0]",0.9974,0.9974,-35.2249,35.4945,70.7194,1.7867,39.5817,51.5532,14,True
2,83,83,1.3833,"[900.0, 333.0]",0.9981,0.9981,-1.0000,-14.0000,15.0000,1.1587,12.9450,18.4349,7,True
3,118,118,1.9667,"[948.0, 324.0]",0.9949,0.9949,4.9500,14.7725,19.7225,1.1450,17.2249,37.3932,3,True
4,170,170,2.8333,"[798.0, 468.0]",0.9971,0.9971,2.0000,-19.0000,21.0000,1.3650,15.3846,10.4915,10,True
5,186,186,3.1000,"[720.0, 735.0]",0.9957,0.9957,-8.0000,33.0000,41.0000,1.7729,23.1257,105.2551,4,True
6,213,213,3.5500,"[627.0, 807.0]",0.9993,0.9993,-15.0000,74.0000,89.0000,1.8829,47.2671,140.4403,5,True
7,261,261,4.3500,"[690.0, 345.0]",0.9975,0.9975,-0.9500,17.3341,18.2841,1.1771,15.5334,171.8699,3,True
8,283,283,4.7167,"[702.0, 225.0]",0.9803,0.9803,-13.1140,-10.6546,23.7685,0.9938,23.9180,14.7084,7,True
9,300,300,5.0000,"[792.0, 255.0]",0.9994,0.9994,10.0000,8.0000,18.0000,1.0396,17.3146,12.5288,1,True


## 5. Export Hit Points JSON


In [6]:
def clean_hit_for_json(item):
    out = dict(item)
    out.pop("confidence_for_score", None)
    return out


def finite_or_none(value, digits=4):
    if value is None or not np.isfinite(value):
        return None
    return round(float(value), digits)


def build_per_frame_scores(score_df, frame_indices, candidate_hits, final_hits):
    candidate_frames = {int(item["frame_index"]) for item in candidate_hits}
    final_frames = {int(item["frame_index"]) for item in final_hits}
    rows = []
    for row_index, row in score_df.iterrows():
        frame_index = int(frame_indices[row_index])
        x = finite_or_none(row["x"], digits=2)
        y = finite_or_none(row["y"], digits=2)
        rows.append({
            "row_index": int(row_index),
            "frame_index": frame_index,
            "time_sec": round(float(frame_index / fps), 4),
            "pixel": [x, y] if x is not None and y is not None else None,
            "confidence": finite_or_none(row["confidence"]),
            "score_x": finite_or_none(row["score_x"]),
            "score_y": finite_or_none(row["score_y"]),
            "abs_total_score": finite_or_none(row["abs_total_score"]),
            "perspective_score_scale": finite_or_none(row["perspective_score_scale"]),
            "adjusted_abs_score": finite_or_none(row["adjusted_abs_score"]),
            "angle_deg": finite_or_none(row["angle_deg"]),
            "is_candidate": frame_index in candidate_frames,
            "is_final_hit": frame_index in final_frames,
        })
    return rows


per_frame_scores = build_per_frame_scores(score_df, frame_indices, candidate_hits, final_hits)


hit_payload = {
    "metadata": {
        "version": "hit-points-mean-reversal-1.0.0",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "input_ball_json": str(BALL_JSON_PATH),
        "input_trajectory_csv": next((str(path) for path in TRAJECTORY_CSV_CANDIDATES if path.exists()), None),
        "track_source": track_source,
        "source_video": metadata.get("source_video"),
        "source_video_path": metadata.get("source_video_path"),
        "fps": fps,
        "video_width": frame_width,
        "video_height": frame_height,
        "total_frames": len(frames),
        "algorithm": "before_after_mean_reversal_without_player_distance",
        "config": HIT_DETECT_CONFIG,
    },
    "summary": {
        "candidate_count": len(candidate_hits),
        "final_hit_count": len(final_hits),
        "final_hit_frames": [int(item["frame_index"]) for item in final_hits],
    },
    "final_hits": [clean_hit_for_json(item) for item in final_hits],
    "candidate_hits": [clean_hit_for_json(item) for item in candidate_hits],
    "per_frame_scores": per_frame_scores,
}

HIT_POINTS_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with HIT_POINTS_JSON_PATH.open("w", encoding="utf-8") as handle:
    json.dump(hit_payload, handle, ensure_ascii=False, indent=2)

print(f"saved: {HIT_POINTS_JSON_PATH}")
hit_payload["summary"]

saved: /Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal.json


{'candidate_count': 149,
 'final_hit_count': 28,
 'final_hit_frames': [8,
  41,
  83,
  118,
  170,
  186,
  213,
  261,
  283,
  300,
  320,
  344,
  359,
  388,
  445,
  540,
  579,
  593,
  657,
  683,
  721,
  735,
  749,
  791,
  809,
  833,
  865,
  893]}

## 6. Render Hit Points Overlay Video


In [7]:
def resolve_source_video_path(metadata):
    candidates = []
    source_path = metadata.get("source_video_path")
    if source_path:
        path = Path(source_path).expanduser()
        candidates.append(path if path.is_absolute() else PROJECT_ROOT / path)
    source_name = metadata.get("source_video")
    if source_name:
        candidates.extend([
            INPUT_DIR / source_name,
            CV_OUTPUT_DIR / source_name,
            BALL_JSON_PATH.parent / source_name,
        ])
    for path in candidates:
        if path.exists():
            return path
    return None


def point_from_track(row_index, scale_x=1.0, scale_y=1.0):
    if row_index < 0 or row_index >= len(track_arr):
        return None
    x, y = track_arr[row_index]
    if not np.isfinite(x) or not np.isfinite(y):
        return None
    return int(round(x * scale_x)), int(round(y * scale_y))


def draw_track_tail(frame, row_index, *, scale_x, scale_y, tail_len=20):
    start = max(0, row_index - tail_len)
    previous = None
    for idx in range(start, row_index + 1):
        point = point_from_track(idx, scale_x, scale_y)
        if point is None:
            previous = None
            continue
        if previous is not None:
            alpha = (idx - start + 1) / max(tail_len, 1)
            color = (int(60 + 80 * alpha), int(180 + 50 * alpha), int(80 + 80 * alpha))
            cv2.line(frame, previous, point, color, 2, cv2.LINE_AA)
        previous = point


def draw_crosshair(frame, point, *, color, radius=18, thickness=2):
    x, y = point
    cv2.circle(frame, point, radius, color, thickness, cv2.LINE_AA)
    cv2.line(frame, (x - radius - 6, y), (x - radius // 2, y), color, thickness, cv2.LINE_AA)
    cv2.line(frame, (x + radius // 2, y), (x + radius + 6, y), color, thickness, cv2.LINE_AA)
    cv2.line(frame, (x, y - radius - 6), (x, y - radius // 2), color, thickness, cv2.LINE_AA)
    cv2.line(frame, (x, y + radius // 2), (x, y + radius + 6), color, thickness, cv2.LINE_AA)


def put_hud(frame, lines):
    overlay = frame.copy()
    height = 30 + 24 * len(lines)
    cv2.rectangle(overlay, (12, 12), (700, height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    for row, text in enumerate(lines):
        cv2.putText(frame, text, (24, 42 + 24 * row), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (245, 245, 245), 1, cv2.LINE_AA)


def render_hit_overlay_video(input_video_path, output_video_path, *, tail_len=20, mark_window=4):
    input_video_path = Path(input_video_path)
    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_video_path.exists():
        raise FileNotFoundError(f"source video not found: {input_video_path}")

    final_by_frame = {int(item["frame_index"]): item for item in final_hits}
    candidate_by_frame = {int(item["frame_index"]): item for item in candidate_hits}
    frame_to_row = {int(frame): row for row, frame in enumerate(frame_indices)}
    last_video_frame = max(frame_indices) if frame_indices else len(track_arr) - 1

    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"could not open source video: {input_video_path}")

    source_fps = cap.get(cv2.CAP_PROP_FPS) or fps
    source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or frame_width)
    source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or frame_height)
    scale_x = source_width / max(frame_width, 1)
    scale_y = source_height / max(frame_height, 1)

    writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        source_fps,
        (source_width, source_height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"could not create output video: {output_video_path}")

    try:
        from tqdm.auto import tqdm
        progress = tqdm(total=last_video_frame + 1, desc="render hit overlay")
    except Exception:
        progress = None

    written = 0
    video_frame = 0
    try:
        while video_frame <= last_video_frame:
            ok, frame = cap.read()
            if not ok:
                break

            row_index = frame_to_row.get(video_frame)
            if row_index is not None:
                draw_track_tail(frame, row_index, scale_x=scale_x, scale_y=scale_y, tail_len=tail_len)
                current_point = point_from_track(row_index, scale_x, scale_y)
                if current_point is not None:
                    cv2.circle(frame, current_point, 5, (120, 220, 120), -1, cv2.LINE_AA)

            active_final = [item for frame_id, item in final_by_frame.items() if abs(frame_id - video_frame) <= mark_window]
            active_candidate = [item for frame_id, item in candidate_by_frame.items() if abs(frame_id - video_frame) <= mark_window]

            for item in active_candidate:
                item_row_index = frame_to_row.get(int(item["frame_index"]))
                if item_row_index is None:
                    continue
                point = point_from_track(item_row_index, scale_x, scale_y)
                if point is not None:
                    cv2.circle(frame, point, 10, (0, 170, 255), 2, cv2.LINE_AA)

            for item in active_final:
                item_row_index = frame_to_row.get(int(item["frame_index"]))
                if item_row_index is None:
                    continue
                point = point_from_track(item_row_index, scale_x, scale_y)
                if point is not None:
                    draw_crosshair(frame, point, color=(40, 40, 255), radius=18, thickness=2)
                    cv2.putText(frame, "HIT", (point[0] + 22, point[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (40, 40, 255), 2, cv2.LINE_AA)

            nearest_final = min(final_hits, key=lambda item: abs(int(item["frame_index"]) - video_frame), default=None)
            if row_index is not None and row_index < len(score_df):
                score_row = score_df.iloc[row_index]
                abs_score = score_row["abs_total_score"]
                adjusted_abs_score = score_row["adjusted_abs_score"]
                score_x = score_row["score_x"]
                score_y = score_row["score_y"]
                angle_deg = score_row["angle_deg"]
                score_text = (
                    f"raw_abs={abs_score:.1f} adj_abs={adjusted_abs_score:.1f} | angle={angle_deg:.1f} | sx={score_x:.1f} sy={score_y:.1f}"
                    if np.isfinite(abs_score) and np.isfinite(adjusted_abs_score) and np.isfinite(angle_deg) and np.isfinite(score_x) and np.isfinite(score_y)
                    else "raw_abs=n/a adj_abs=n/a | angle=n/a | sx=n/a sy=n/a"
                )
            else:
                score_text = "raw_abs=n/a adj_abs=n/a | angle=n/a | sx=n/a sy=n/a"
            hud_lines = [
                f"frame: {video_frame} | track: {track_source}",
                score_text,
                f"final hits: {len(final_hits)} | candidates: {len(candidate_hits)}",
            ]
            if nearest_final is not None and abs(int(nearest_final["frame_index"]) - video_frame) <= mark_window:
                hud_lines.append(
                    f"active hit frame={nearest_final['frame_index']} adj={nearest_final['adjusted_abs_score']:.1f} raw={nearest_final['abs_total_score']:.1f} angle={nearest_final['angle_deg']:.1f}"
                )
            else:
                hud_lines.append("orange=candidate | red=final hit")
            put_hud(frame, hud_lines)

            writer.write(frame)
            written += 1
            video_frame += 1
            if progress is not None:
                progress.update(1)
    finally:
        if progress is not None:
            progress.close()
        cap.release()
        writer.release()

    return {
        "input_video": str(input_video_path),
        "output_video": str(output_video_path),
        "frames_written": written,
        "source_fps": source_fps,
        "source_size": (source_width, source_height),
    }


SOURCE_VIDEO_PATH = resolve_source_video_path(metadata)
if SOURCE_VIDEO_PATH is None:
    raise FileNotFoundError("Could not infer source video path from metadata. Set SOURCE_VIDEO_PATH manually.")

render_hit_overlay_video(SOURCE_VIDEO_PATH, HIT_OVERLAY_VIDEO_PATH, tail_len=20, mark_window=4)


/Users/mac/miniconda3/envs/bst_main/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
render hit overlay: 100%|██████████| 929/929 [00:06<00:00, 147.12it/s]


{'input_video': '/Users/mac/PycharmProjects/tennis_data_master/cv/data/input/long4.mp4',
 'output_video': '/Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal_overlay.mp4',
 'frames_written': 929,
 'source_fps': 60.0,
 'source_size': (1920, 1080)}

## 7. Quick Inspect


In [8]:
print("json:", HIT_POINTS_JSON_PATH)
print("video:", HIT_OVERLAY_VIDEO_PATH)
print("final hit frames:", [item["frame_index"] for item in final_hits])
pd.DataFrame(candidate_hits).sort_values("abs_total_score", ascending=False).head(30)


json: /Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal.json
video: /Users/mac/PycharmProjects/tennis_data_master/cv/data/output/hit_points_mean_reversal_overlay.mp4
final hit frames: [8, 41, 83, 118, 170, 186, 213, 261, 283, 300, 320, 344, 359, 388, 445, 540, 579, 593, 657, 683, 721, 735, 749, 791, 809, 833, 865, 893]


,row_index,frame_index,time_sec,pixel,confidence,confidence_for_score,score_x,score_y,abs_total_score,perspective_score_scale,adjusted_abs_score,angle_deg,is_final_hit
41,213,213,3.5500,"[627.0, 807.0]",0.9993,0.9993,-15.0000,74.0000,89.0000,1.8829,47.2671,140.4403,True
140,893,893,14.8833,"[1086.0, 600.0]",0.9992,0.9992,18.6381,67.4872,86.1253,1.5667,54.9736,103.6863,True
67,388,388,6.4667,"[1308.0, 753.0]",0.9992,0.9992,15.0000,56.0000,71.0000,1.8004,39.4353,139.7636,True
7,41,41,0.6833,"[798.0, 744.0]",0.9974,0.9974,-35.2249,35.4945,70.7194,1.7867,39.5817,51.5532,True
126,833,833,13.8833,"[732.0, 558.0]",0.9953,0.9953,-24.8974,41.3576,66.2550,1.5025,44.0965,60.5622,True
139,892,892,14.8667,"[1071.0, 582.0]",0.9937,0.9937,11.4556,49.2361,60.6917,1.5392,39.4315,122.6615,False
128,835,835,13.9167,"[732.0, 546.0]",0.9889,0.9889,-19.8974,29.3576,49.2550,1.4842,33.1870,43.6462,False
6,40,40,0.6667,"[804.0, 732.0]",0.9981,0.9981,-26.6138,17.8772,44.4910,1.7683,25.1598,154.0520,False
56,320,320,5.3333,"[888.0, 321.0]",0.9965,0.9965,-24.0000,-18.0000,42.0000,1.1404,36.8286,18.4349,True
127,834,834,13.9000,"[755.69, 535.93]",0.9943,0.9943,35.3843,-6.1457,41.5299,1.4688,28.2752,161.1616,False


In [ ]:
def _is_valid_point(point):
    return point is not None and len(point) >= 2 and np.isfinite(point[0]) and np.isfinite(point[1])


def detect_landing_candidates(score_df, frame_indices, *, prev_window=3, next_window=3, min_abs_total_score=8.0, min_angle_deg=10.0, min_vertical_score=8.0, group_gap=12):
    candidates = []
    y_values = score_df["y"].to_numpy(dtype=np.float64)
    for row_index in range(prev_window, len(score_df) - next_window):
        row = score_df.iloc[row_index]
        if not np.isfinite(row["y"]):
            continue
        prev_vals = y_values[row_index - prev_window:row_index]
        next_vals = y_values[row_index + 1:row_index + 1 + next_window]
        if len(prev_vals) < prev_window or len(next_vals) < next_window:
            continue
        if not np.all(np.isfinite(prev_vals)) or not np.all(np.isfinite(next_vals)):
            continue
        vertical_score = (row["y"] - float(np.mean(prev_vals))) + (row["y"] - float(np.mean(next_vals)))
        if vertical_score < min_vertical_score:
            continue
        confidence = float(row["confidence"]) if np.isfinite(row["confidence"]) else np.nan
        if np.isfinite(confidence) and confidence < 0.0:
            continue
        if row["adjusted_abs_score"] < min_abs_total_score:
            continue
        if row["angle_deg"] < min_angle_deg:
            continue
        candidates.append({
            "row_index": int(row_index),
            "frame_index": int(frame_indices[row_index]),
            "time_sec": round(float(frame_indices[row_index] / fps), 4),
            "pixel": [round(float(row["x"]), 2), round(float(row["y"]), 2)],
            "confidence": round(confidence, 4) if np.isfinite(confidence) else None,
            "confidence_for_score": float(np.clip(confidence, 0.0, 1.0)) if np.isfinite(confidence) else 0.5,
            "score_x": round(float(row["score_x"]), 4),
            "score_y": round(float(row["score_y"]), 4),
            "abs_total_score": round(float(row["abs_total_score"]), 4),
            "perspective_score_scale": round(float(row["perspective_score_scale"]), 4),
            "adjusted_abs_score": round(float(row["adjusted_abs_score"]), 4),
            "angle_deg": round(float(row["angle_deg"]), 4),
            "breakpoint_type": "landing",
            "vertical_score": round(float(vertical_score), 4),
            "priority": 1,
        })
    return group_candidates(candidates, group_gap=group_gap)


def merge_breakpoints(hit_candidates, landing_candidates, *, group_gap=12):
    combined = []
    for item in hit_candidates:
        row = dict(item)
        row["breakpoint_type"] = row.get("breakpoint_type", "hit")
        row["priority"] = int(row.get("priority", 2))
        combined.append(row)
    for item in landing_candidates:
        row = dict(item)
        row["breakpoint_type"] = row.get("breakpoint_type", "landing")
        row["priority"] = int(row.get("priority", 1))
        combined.append(row)
    if not combined:
        return []
    combined = sorted(combined, key=lambda row: row["frame_index"])
    groups = [[combined[0]]]
    for item in combined[1:]:
        if item["frame_index"] - groups[-1][-1]["frame_index"] < group_gap:
            groups[-1].append(item)
        else:
            groups.append([item])
    merged = []
    for group in groups:
        best = max(group, key=lambda row: (row.get("priority", 0), float(row.get("adjusted_abs_score") or 0.0), float(row.get("vertical_score") or 0.0)))
        best = dict(best)
        best["group_size"] = len(group)
        merged.append(best)
    return merged


def build_segment_bounds(total_frames, breakpoint_frames):
    breakpoint_frames = sorted({int(frame) for frame in breakpoint_frames if 0 <= int(frame) < total_frames - 1})
    starts = [0]
    ends = []
    for frame in breakpoint_frames:
        ends.append(frame)
        starts.append(frame + 1)
    ends.append(total_frames - 1)
    bounds = []
    for start, end in zip(starts, ends):
        if start <= end:
            bounds.append((start, end))
    return bounds


def ransac_parabola(frames, xs, ys, threshs, n_iters, deg_x, deg_y):
    n_pts = len(frames)
    min_points = max(deg_x + 1, deg_y + 1, 3)
    if n_pts < min_points:
        return None, 0, None

    t = np.asarray(frames, dtype=np.float64)
    x_arr = np.asarray(xs, dtype=np.float64)
    y_arr = np.asarray(ys, dtype=np.float64)
    th_arr = np.asarray(threshs, dtype=np.float64)
    valid = np.isfinite(t) & np.isfinite(x_arr) & np.isfinite(y_arr) & np.isfinite(th_arr)
    t, x_arr, y_arr, th_arr = t[valid], x_arr[valid], y_arr[valid], th_arr[valid]
    n_pts = len(t)
    if n_pts < min_points:
        return None, 0, None

    sample_size = min(n_pts, max(deg_x + 1, deg_y + 1) + 1)
    best_inliers = 0
    best_cx = None
    best_cy = None
    best_mask = None
    rng = np.random.RandomState(42)

    for _ in range(n_iters):
        idx = rng.choice(n_pts, sample_size, replace=False)
        try:
            cx = np.polyfit(t[idx], x_arr[idx], deg_x)
            cy = np.polyfit(t[idx], y_arr[idx], deg_y)
        except (np.linalg.LinAlgError, ValueError):
            continue
        residuals = np.sqrt((x_arr - np.polyval(cx, t)) ** 2 + (y_arr - np.polyval(cy, t)) ** 2)
        mask = residuals < th_arr
        n_inliers = int(mask.sum())
        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_cx = cx
            best_cy = cy
            best_mask = mask

    if best_mask is None or best_inliers < min_points:
        return None, best_inliers, best_mask

    try:
        best_cx = np.polyfit(t[best_mask], x_arr[best_mask], deg_x)
        best_cy = np.polyfit(t[best_mask], y_arr[best_mask], deg_y)
    except (np.linalg.LinAlgError, ValueError):
        pass
    return (best_cx, best_cy), best_inliers, best_mask


def predict_parabola(params, t_rel):
    if params is None:
        return np.nan, np.nan
    cx, cy = params
    if cx is None or cy is None:
        return np.nan, np.nan
    return float(np.polyval(cx, t_rel)), float(np.polyval(cy, t_rel))


def segment_ransac_process(track_arr, confidences, frame_indices, breakpoint_frames, *, config):
    total_frames = len(track_arr)
    segments = build_segment_bounds(total_frames, breakpoint_frames)
    rows = []
    for segment_id, (start, end) in enumerate(segments):
        for frame_index in range(start, end + 1):
            window_start = max(start, frame_index - config["window_half"])
            window_end = min(end, frame_index + config["window_half"])
            frames = []
            xs = []
            ys = []
            threshs = []
            for j in range(window_start, window_end + 1):
                if j == frame_index:
                    continue
                point = track_arr[j]
                if not _is_valid_point(point):
                    continue
                t = j - frame_index if config["t_normalize"] else j
                py = float(point[1])
                cf = confidences[j] if j < len(confidences) else np.nan
                frames.append(t)
                xs.append(float(point[0]))
                ys.append(py)
                threshs.append(config["base_thresh"] * perspective_score_scale(py, frame_height, far_scale=config["far_scale"], near_scale=config["near_scale"]))
            raw_point = track_arr[frame_index]
            raw_x, raw_y = (float(raw_point[0]), float(raw_point[1])) if _is_valid_point(raw_point) else (np.nan, np.nan)
            if len(frames) < config["min_points"]:
                rows.append({
                    "frame": int(frame_index),
                    "segment_id": int(segment_id),
                    "segment_start_frame": int(start),
                    "segment_end_frame": int(end),
                    "raw_x": raw_x,
                    "raw_y": raw_y,
                    "current_x": raw_x,
                    "current_y": raw_y,
                    "ransac_event_x": raw_x,
                    "ransac_event_y": raw_y,
                    "ransac_smooth_x": raw_x,
                    "ransac_smooth_y": raw_y,
                    "ransac_status": "raw_kept" if np.isfinite(raw_x) and np.isfinite(raw_y) else "gap",
                    "ransac_residual_px": np.nan,
                    "ransac_threshold_px": np.nan,
                    "ransac_inlier_count": len(frames),
                    "breakpoint_frame": int(frame_index) if frame_index in breakpoint_frames else None,
                    "breakpoint_type": next((item.get("breakpoint_type") for item in breakpoint_index_map.get(frame_index, [])), None),
                })
                continue

            params, n_inliers, _ = ransac_parabola(
                frames,
                xs,
                ys,
                threshs,
                n_iters=config["iters"],
                deg_x=config["deg_x"],
                deg_y=config["deg_y"],
            )
            if params is None:
                rows.append({
                    "frame": int(frame_index),
                    "segment_id": int(segment_id),
                    "segment_start_frame": int(start),
                    "segment_end_frame": int(end),
                    "raw_x": raw_x,
                    "raw_y": raw_y,
                    "current_x": raw_x,
                    "current_y": raw_y,
                    "ransac_event_x": raw_x,
                    "ransac_event_y": raw_y,
                    "ransac_smooth_x": raw_x,
                    "ransac_smooth_y": raw_y,
                    "ransac_status": "raw_kept" if np.isfinite(raw_x) and np.isfinite(raw_y) else "gap",
                    "ransac_residual_px": np.nan,
                    "ransac_threshold_px": np.nan,
                    "ransac_inlier_count": len(frames),
                    "breakpoint_frame": int(frame_index) if frame_index in breakpoint_frames else None,
                    "breakpoint_type": next((item.get("breakpoint_type") for item in breakpoint_index_map.get(frame_index, [])), None),
                })
                continue

            t_self = 0.0 if config["t_normalize"] else float(frame_index)
            px, py = predict_parabola(params, t_self)
            if np.isfinite(raw_y):
                cur_y, cur_cf = raw_y, confidences[frame_index] if frame_index < len(confidences) else np.nan
            else:
                cur_y, cur_cf = frame_height / 2, np.nan
            dyn_thresh = config["base_thresh"] * perspective_score_scale(cur_y, frame_height, far_scale=config["far_scale"], near_scale=config["near_scale"])
            residual = float(np.hypot(raw_x - px, raw_y - py)) if np.isfinite(raw_x) and np.isfinite(raw_y) else np.nan
            event_x, event_y = (raw_x, raw_y) if np.isfinite(raw_x) and np.isfinite(raw_y) and residual < dyn_thresh else (px, py)
            status = "inlier" if np.isfinite(raw_x) and np.isfinite(raw_y) and residual < dyn_thresh else ("outlier_corrected" if np.isfinite(raw_x) and np.isfinite(raw_y) else "winransac_interp")
            rows.append({
                "frame": int(frame_index),
                "segment_id": int(segment_id),
                "segment_start_frame": int(start),
                "segment_end_frame": int(end),
                "raw_x": raw_x,
                "raw_y": raw_y,
                "current_x": px,
                "current_y": py,
                "ransac_event_x": event_x,
                "ransac_event_y": event_y,
                "ransac_smooth_x": px,
                "ransac_smooth_y": py,
                "ransac_status": status,
                "ransac_residual_px": residual,
                "ransac_threshold_px": dyn_thresh,
                "ransac_inlier_count": int(n_inliers),
                "breakpoint_frame": int(frame_index) if frame_index in breakpoint_frames else None,
                "breakpoint_type": next((item.get("breakpoint_type") for item in breakpoint_index_map.get(frame_index, [])), None),
            })
    return pd.DataFrame(rows)


landing_candidates = detect_landing_candidates(
    score_df,
    frame_indices,
    prev_window=HIT_DETECT_CONFIG.get("prev_window", 3),
    next_window=HIT_DETECT_CONFIG.get("next_window", 3),
    min_abs_total_score=HIT_DETECT_CONFIG.get("min_abs_total_score", 8.0),
    min_angle_deg=HIT_DETECT_CONFIG.get("min_angle_deg", 10.0),
    min_vertical_score=HIT_DETECT_CONFIG.get("min_abs_total_score", 8.0),
    group_gap=HIT_DETECT_CONFIG.get("group_gap", 12),
)
merged_breakpoints = merge_breakpoints(candidate_hits, landing_candidates, group_gap=HIT_DETECT_CONFIG.get("group_gap", 12))
breakpoint_frames = [int(item["frame_index"]) for item in merged_breakpoints]
breakpoint_index_map = {}
for item in merged_breakpoints:
    breakpoint_index_map.setdefault(int(item["frame_index"]), []).append(item)

SEGMENT_RANSAC_CONFIG = {
    "window_half": 10,
    "iters": 80,
    "base_thresh": 16.0,
    "deg_x": 1,
    "deg_y": 2,
    "t_normalize": True,
    "far_scale": 0.65,
    "near_scale": 2.30,
    "min_points": 4,
}

segmented_ransac_df = segment_ransac_process(track_arr, raw_confidences, frame_indices, breakpoint_frames, config=SEGMENT_RANSAC_CONFIG)
SEGMENTED_RANSAC_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
segmented_ransac_df.to_csv(SEGMENTED_RANSAC_CSV_PATH, index=False)

print(f"breakpoints={len(breakpoint_frames)} -> {breakpoint_frames[:20]}")
print(f"segmented ransac saved: {SEGMENTED_RANSAC_CSV_PATH}")
segmented_ransac_df.head(5)